# Cox Proportional Hazard Modeling
Beginning from the most simple clinical metadata, we will test Cox Proportional Hazards Modeling to predict cancer survival

In [1]:
import pandas as pd
from lifelines import CoxPHFitter
import pyhere as here

In [2]:
here.here()

PosixPath('/Users/jmakings/Documents/Projects/breast_cancer_survival_prediction')

In [3]:
# load clinical metadata with PCA + UMAP features
pca_clinical_df = pd.read_csv(here.here("data", "processed","clinical_dimred_features.csv"))

In [4]:
pca_clinical_df

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,PC293_harmony,PC294_harmony,PC295_harmony,PC296_harmony,PC297_harmony,PC298_harmony,PC299_harmony,PC300_harmony,UMAP1,UMAP2
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0.244572,-0.496506,0.168578,0.533063,0.434756,0.272053,0.324875,-0.103091,-1.800003,-6.149750
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0.303471,-0.844191,-0.596785,0.820834,0.064187,-0.056478,0.407295,0.292778,-2.570254,-2.240858
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0.059991,-0.518747,0.594953,0.316483,0.071531,0.090688,0.333440,0.199574,-5.767281,-4.865173
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,-0.689182,0.091264,-0.301379,-0.521918,0.213592,-0.737401,1.100436,-0.762680,-5.721486,-4.890594
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,-0.590699,-0.293891,1.523677,-1.229448,0.036228,0.395598,0.133042,0.194311,-1.569188,-1.622840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,7295,43.10,BREAST CONSERVING,Breast Cancer,Breast Invasive Lobular Carcinoma,High,0,LumA,4.0,Positve,...,-0.457317,0.608927,0.001513,-0.051347,0.044357,-0.138593,-0.374052,0.445042,-1.645502,-3.813061
1900,7296,42.88,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,...,0.176880,0.559336,0.212478,0.075198,-0.336113,-0.213558,0.347952,0.099582,-1.342959,-1.388248
1901,7297,62.90,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,...,-0.460358,0.201798,-0.437061,0.109682,-0.401253,-0.174120,-0.289600,0.339712,-3.293207,-0.444603
1902,7298,61.16,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,Moderate,0,LumB,4.0,Positve,...,0.399676,-0.208885,0.881920,0.048649,0.519081,-0.593877,-0.908927,0.127365,-3.406587,-0.689774


### Preprocess data for cox model functions (Drop missing values and One-hot encoding)

In [35]:
def preprocess_for_cox(df, duration_col, event_col):
    # Drop rows with missing values in duration or event columns
    df = df.dropna(subset=[duration_col, event_col])
    
    # Ensure duration column is numeric
    df[duration_col] = pd.to_numeric(df[duration_col], errors='coerce')
    
    # Drop rows with non-numeric duration values
    df = df.dropna(subset=[duration_col])
    
    return df

In [56]:
# converts float columns with whole numbers to integer type
def convert_whole_float_columns_to_int(df):
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            # Check if all non-null values are whole numbers
            if df[col].dropna().apply(float.is_integer).all():
                df[col] = df[col].astype("Int64")  # nullable integer type
    return df

In [72]:
def one_hot_encode(df, convert_whole_num_ints=True):
    # One hot encode categorical variables

    # Identify categorical columns automatically
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    # Apply one-hot encoding only to categorical columns
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # Change boolean columns to integers
    bool_cols = df_encoded.select_dtypes(include=["bool"]).columns
    df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)
    
    if convert_whole_num_ints:
        df_encoded = convert_whole_float_columns_to_int(df_encoded)
    return df_encoded

In [58]:
# One hot encode categorical variables

df_encoded = one_hot_encode(pca_clinical_df)

In [59]:
df_encoded.dtypes.value_counts()

float64    310
int64       62
Name: count, dtype: int64

In [60]:
df_encoded = convert_whole_float_columns_to_int(df_encoded)

In [65]:
df_encoded.dtypes

patient_id                                  int64
age_at_diagnosis                          float64
chemotherapy                                int64
cohort                                      Int64
neoplasm_histologic_grade                   Int64
                                           ...   
tumor_stage_3.0                             int64
tumor_stage_4.0                             int64
tumor_stage_unknown                         int64
death_from_cancer_Died of Other Causes      int64
death_from_cancer_Living                    int64
Length: 372, dtype: object

In [18]:
# save one-hot encoded clinical metadata with PCA + UMAP features
df_encoded.to_csv(here.here("data", "processed","clinical_dimred_features_onehot.csv"), index=False)

## Cox Proportional Hazards Model - Limited Clinical Data
#### We will start with only a few strong clinical features, and expand from there

In [67]:
# start with strongest clinical features only
baseline_cols = [
    "overall_survival_months",   # time
    "death_from_cancer",         # event
    "age_at_diagnosis",          
    "tumor_size",
    "tumor_stage",
    "lymph_nodes_examined_positive",
    "neoplasm_histologic_grade",
    "er_status",
    "pr_status",
    "her2_status"
]


In [73]:
# subset for cox model 
df_cox = pca_clinical_df[baseline_cols]

# preprocess for cox model
df_cox = preprocess_for_cox(df_cox, duration_col="overall_survival_months", event_col="death_from_cancer")
df_cox = one_hot_encode(df_cox)

/var/folders/lg/ycvb06q90x9cty7h1176t0200000gn/T/ipykernel_91736/1554145649.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[duration_col] = pd.to_numeric(df[duration_col], errors='coerce')


In [76]:
pca_clinical_df["death_from_cancer"].value_counts()

death_from_cancer
Living                  801
Died of Disease         622
Died of Other Causes    480
Name: count, dtype: int64

In [ ]:
# death from cancer value counts after one-hot encoding
# we also have "Died from Other Causes", which we will treat as censored
df_cox[[ "death_from_cancer_Died of Other Causes","death_from_cancer_Living"]].value_counts()

death_from_cancer_Died of Other Causes  death_from_cancer_Living
0                                       1                           801
                                        0                           622
1                                       0                           480
Name: count, dtype: int64

In [77]:
df_cox

,overall_survival_months,age_at_diagnosis,tumor_size,lymph_nodes_examined_positive,neoplasm_histologic_grade,death_from_cancer_Died of Other Causes,death_from_cancer_Living,tumor_stage_1.0,tumor_stage_2.0,tumor_stage_3.0,tumor_stage_4.0,tumor_stage_unknown,er_status_Positive,pr_status_Positive,her2_status_Positive
0,140.500000,75.65,22.0,10,3,0,1,0,1,0,0,0,1,0,0
1,84.633333,43.19,10.0,0,3,0,1,1,0,0,0,0,1,1,0
2,163.700000,48.87,15.0,1,2,0,0,0,1,0,0,0,1,1,0
3,164.933333,47.68,25.0,3,2,0,1,0,1,0,0,0,1,1,0
4,41.366667,76.97,40.0,8,3,0,0,0,1,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,196.866667,43.10,25.0,1,3,0,1,0,0,0,0,1,1,1,0
1900,44.733333,42.88,20.0,1,3,0,0,0,0,0,0,1,1,0,1
1901,175.966667,62.90,25.0,45,3,0,0,0,0,0,0,1,1,1,0
1902,86.233333,61.16,25.0,12,2,1,0,0,0,0,0,1,1,1,0


In [80]:
# Rename for clarity
df_cox = df_cox.rename(columns={
    "overall_survival_months": "time",
    "death_from_cancer_Living": "event"
})
# Drop the "Died of Other Causes" column as we are treating it as censored
df_cox = df_cox.drop(columns=["death_from_cancer_Died of Other Causes"])
# Drop missing values (simple first version)
df_cox = df_cox.dropna()


In [81]:
df_cox

,time,age_at_diagnosis,tumor_size,lymph_nodes_examined_positive,neoplasm_histologic_grade,event,tumor_stage_1.0,tumor_stage_2.0,tumor_stage_3.0,tumor_stage_4.0,tumor_stage_unknown,er_status_Positive,pr_status_Positive,her2_status_Positive
0,140.500000,75.65,22.0,10,3,1,0,1,0,0,0,1,0,0
1,84.633333,43.19,10.0,0,3,1,1,0,0,0,0,1,1,0
2,163.700000,48.87,15.0,1,2,0,0,1,0,0,0,1,1,0
3,164.933333,47.68,25.0,3,2,1,0,1,0,0,0,1,1,0
4,41.366667,76.97,40.0,8,3,0,0,1,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,196.866667,43.10,25.0,1,3,1,0,0,0,0,1,1,1,0
1900,44.733333,42.88,20.0,1,3,0,0,0,0,0,1,1,0,1
1901,175.966667,62.90,25.0,45,3,0,0,0,0,0,1,1,1,0
1902,86.233333,61.16,25.0,12,2,0,0,0,0,0,1,1,1,0


In [82]:
# fit Cox Proportional Hazards model
cph = CoxPHFitter()
cph.fit(df_cox, duration_col="time", event_col="event")

cph.print_summary()


/Users/jmakings/miniconda3/envs/bc_survival/lib/python3.11/site-packages/lifelines/fitters/coxph_fitter.py:1607: ConvergenceWarning: Newton-Raphson convergence completed successfully but norm(delta) is still high, 0.828. This may imply non-unique solutions to the maximum likelihood. Perhaps there is collinearity or complete separation in the dataset?

  warnings.warn(


<lifelines.CoxPHFitter: fitted with 1814 total observations, 1040 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1814
number of events observed = 774
   partial log-likelihood = -4719.60
         time fit was run = 2025-12-10 02:28:05 UTC

---
                               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                      
age_at_diagnosis              -0.01      0.99      0.00           -0.02           -0.00                0.99                1.00
tumor_size                    -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
lymph_nodes_examined_positive  0.02      1.02      0.01           -0.01            0.04                0.99                1.04
neoplasm_histologic_grade     -0.13      0.87      0.06           -0.25           -0.01                0.78                0.99
tumor_stage_1.0               12.26  2.11e+05    904.17        -1759.88         1784.40                0.00                 inf
tumor_stage_2.0               12.42  2.48e+05    904.17        -1759.71         1784.56                0.00                 inf
tumor_stage_3.0               12.55  2.81e+05    904.17        -1759.59         1784.69                0.00                 inf
tumor_stage_4.0               12.54  2.79e+05    904.17        -1759.60         1784.68                0.00                 inf
tumor_stage_unknown           12.52  2.73e+05    904.17        -1759.62         1784.66                0.00                 inf
er_status_Positive            -0.06      0.94      0.11           -0.29            0.16                0.75                1.18
pr_status_Positive            -0.13      0.87      0.09           -0.31            0.05                0.73                1.05
her2_status_Positive           0.24      1.27      0.12           -0.00            0.48                1.00                1.61

                               cmp to     z    p  -log2(p)
covariate                                                 
age_at_diagnosis                 0.00 -2.71 0.01      7.23
tumor_size                       0.00 -1.03 0.30      1.72
lymph_nodes_examined_positive    0.00  1.23 0.22      2.18
neoplasm_histologic_grade        0.00 -2.19 0.03      5.11
tumor_stage_1.0                  0.00  0.01 0.99      0.02
tumor_stage_2.0                  0.00  0.01 0.99      0.02
tumor_stage_3.0                  0.00  0.01 0.99      0.02
tumor_stage_4.0                  0.00  0.01 0.99      0.02
tumor_stage_unknown              0.00  0.01 0.99      0.02
er_status_Positive               0.00 -0.55 0.58      0.78
pr_status_Positive               0.00 -1.45 0.15      2.78
her2_status_Positive             0.00  1.96 0.05      4.31
---
Concordance = 0.57
Partial AIC = 9463.20
log-likelihood ratio test = 31.31 on 12 df
-log2(p) of ll-ratio test = 9.14

### We are getting convergence warnings due to Tumor Stage having very unbalanced data: it is very correlated with cancer death

In [86]:
pd.crosstab(pca_clinical_df["tumor_stage"], pca_clinical_df["death_from_cancer"])


death_from_cancer,Died of Disease,Died of Other Causes,Living
tumor_stage,,,
0.0,0,1,3
1.0,108,106,260
2.0,286,196,318
3.0,67,19,29
4.0,8,0,1
unknown,153,158,190


### Let's drop the Tumor Stage column and try again

In [ ]:
df_cox = df_cox.drop(columns=[col for col in df_cox.columns if "tumor_stage" in col])

cph = CoxPHFitter()
cph.fit(df_cox, duration_col="time", event_col="event")
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 1814 total observations, 1040 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1814
number of events observed = 774
   partial log-likelihood = -4723.04
         time fit was run = 2025-12-10 02:35:09 UTC

---
                               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                      
age_at_diagnosis              -0.01      0.99      0.00           -0.01           -0.00                0.99                1.00
tumor_size                    -0.00      1.00      0.00           -0.01            0.00                0.99                1.01
lymph_nodes_examined_positive  0.02      1.03      0.01            0.00            0.05                1.00                1.05
neoplasm_histologic_grade     -0.13      0.88      0.06           -0.25           -0.01                0.78                0.99
er_status_Positive            -0.10      0.90      0.11           -0.32            0.12                0.72                1.13
pr_status_Positive            -0.12      0.88      0.09           -0.30            0.06                0.74                1.06
her2_status_Positive           0.24      1.28      0.12            0.00            0.48                1.00                1.62

                               cmp to     z    p  -log2(p)
covariate                                                 
age_at_diagnosis                 0.00 -2.49 0.01      6.29
tumor_size                       0.00 -0.25 0.81      0.31
lymph_nodes_examined_positive    0.00  2.09 0.04      4.79
neoplasm_histologic_grade        0.00 -2.16 0.03      5.04
er_status_Positive               0.00 -0.88 0.38      1.41
pr_status_Positive               0.00 -1.34 0.18      2.46
her2_status_Positive             0.00  2.00 0.05      4.46
---
Concordance = 0.56
Partial AIC = 9460.09
log-likelihood ratio test = 24.41 on 7 df
-log2(p) of ll-ratio test = 10.02

## This is a good baseline model, now we will seek to improve it